In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
from math import exp

In [13]:
df = pd.read_csv("../data/BTC-USD_news.csv")
df.head()

,Tytuł,Źródło,Data,Link
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...


In [14]:
sentiment_model = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [15]:
titles = df["Tytuł"].tolist()
results = sentiment_model(titles)
df["sentiment_label"] = [result["label"] for result in results]
df["sentiment_score"] = [result["score"] for result in results]
df["sentiment_score"] = df["sentiment_score"].round(4)
df.head(10)

,Tytuł,Źródło,Data,Link,sentiment_label,sentiment_score
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...,POSITIVE,0.9940
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9942
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9975
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9997
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9982
5,Bitcoin Drops 4.3% as $14 Billion Options Expi...,finance.yahoo.com,2026-03-27 21:00,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9994
6,Bitcoin Policy Institute Voices Strong Opposit...,bankless.com,2026-03-27 19:47,https://www.bankless.com/read/news/bitcoin-pol...,POSITIVE,0.7277
7,Analyst warns Bitcoin traders are buying ‘cras...,thestreet.com,2026-03-27 19:30,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9901
8,Bitcoin extends slide as options point toward ...,finance.yahoo.com,2026-03-27 19:20,https://finance.yahoo.com/news/bitcoin-slumps-...,NEGATIVE,0.9913
9,Sector Update: Financial Stocks Decline Friday...,finance.yahoo.com,2026-03-27 19:04,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9987


In [ ]:
# Kierunek sentymentu + siła sentymentu

def callculate_sentiment_score(row):
    if row["sentiment_label"] == "POSITIVE":
        return row["sentiment_score"]
    elif row["sentiment_label"] == "NEGATIVE":
        return -row["sentiment_score"]
    else:
        return 0.0

# Warygodność żródła
SOURCE_RELIABILITY = {
    "finance.yahoo.com": 1.0,
    "thestreet.com": 0.9,
    "bankless.com": 0.1,
}
DEFAULT_RELIABILITY = 0.5

def get_source_reliability(source):
    return SOURCE_RELIABILITY.get(source, DEFAULT_RELIABILITY)

# Aktualność informacji
def calculate_freshness_score(pub_date):
    current_time = pd.Timestamp.now()
    pub_time = pd.to_datetime(pub_date)
    time_diff = (current_time - pub_time).total_seconds() / 3600
    
    return exp(-0.05 * time_diff)


# score = sentiment_score × (w1 + w2 × reliability) + sign(sentiment) × w3 × freshness
def calculate_importance_score(row):
    sentiment_score = callculate_sentiment_score(row)
    source_reliability = get_source_reliability(row["Źródło"])
    freshness_score = calculate_freshness_score(row["Data"])
    w1, w2, w3 = 0.5, 0.3, 0.2
    sign = 0 if sentiment_score == 0 else (1 if sentiment_score > 0 else -1)
    importance_score = sentiment_score * (w1 + w2 * source_reliability) + sign * w3 * freshness_score
    return round(importance_score, 2)



# df["importance_score"] = df.apply(calculate_importance_score, axis=1)

# sorted_df = df.sort_values(by="importance_score", key=abs , ascending=False)
# sorted_df.head(10)

,Tytuł,Źródło,Data,Link,sentiment_label,sentiment_score,importance_score
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9997,-0.88
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9975,-0.88
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9982,-0.88
5,Bitcoin Drops 4.3% as $14 Billion Options Expi...,finance.yahoo.com,2026-03-27 21:00,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9994,-0.88
9,Sector Update: Financial Stocks Decline Friday...,finance.yahoo.com,2026-03-27 19:04,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9987,-0.87
8,Bitcoin extends slide as options point toward ...,finance.yahoo.com,2026-03-27 19:20,https://finance.yahoo.com/news/bitcoin-slumps-...,NEGATIVE,0.9913,-0.87
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9942,-0.85
7,Analyst warns Bitcoin traders are buying ‘cras...,thestreet.com,2026-03-27 19:30,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9901,-0.84
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...,POSITIVE,0.9940,0.80
6,Bitcoin Policy Institute Voices Strong Opposit...,bankless.com,2026-03-27 19:47,https://www.bankless.com/read/news/bitcoin-pol...,POSITIVE,0.7277,0.46


In [22]:
def summarize_signals(df, symbol, top_n):
    df = df.copy()
    df["importance_score"] = df.apply(calculate_importance_score, axis=1)
    sorted_df = df.sort_values(by="importance_score", key=abs , ascending=False)
    
    total_score = sorted_df["importance_score"].sum()
    avg_score = sorted_df["importance_score"].mean()

    if avg_score > 0.3:
        signal = "KUP"
        signal_desc = "Przewaga pozytywnych sygnałów, sugeruje potencjalny wzrost ceny."
    elif avg_score > 0.05:
        signal = "OBSERWUJ"
        signal_desc = "Słaba przewaga pozytywnych sygnałów, sugeruje ostrożność i obserwację rynku."
    elif avg_score < -0.3:
        signal = "SPRZEDAJ"
        signal_desc = "Przewaga negatywnych sygnałów, sugeruje potencjalny spadek ceny."
    elif avg_score < -0.05:
        signal = "OBSERWUJ"
        signal_desc = "Słaba przewaga negatywnych sygnałów, sugeruje ostrożność i obserwację rynku."
    else:
        signal = "Neutralny"
        signal_desc = "Brak wyraźnej przewagi sygnałów, sugeruje neutralną pozycję."

    top_news = sorted_df.head(top_n)[["Tytuł", "Źródło", "Data", "importance_score"]]

    print(f"Sygnal dla {symbol}: {signal}")
    print(f"Opis sygnału: {signal_desc}")
    print(f"Avg Score: {avg_score:.4f}, Total Score: {total_score:.4f} | Newsów: {len(df)}")

    print(f"Top {top_n} newsów:")
    for _, row in top_news.iterrows():
        direction = "^^^" if row["importance_score"] > 0 else "vvv"
        print(f"{direction} {row['Tytuł']} ({row['Źródło']}, {row['Data']}) - Score: {row['importance_score']}")

summarize_signals(df, "BTC-USD", 5)


Sygnal dla BTC-USD: SPRZEDAJ
Opis sygnału: Przewaga negatywnych sygnałów, sugeruje potencjalny spadek ceny.
Avg Score: -0.5690, Total Score: -5.6900 | Newsów: 10
Top 5 newsów:
vvv Top Cryptocurrencies Fall; Bitcoin Falls Below $67,000 (finance.yahoo.com, 2026-03-27 21:04) - Score: -0.88
vvv Crypto Thefts Hit $2.7 Billion as Coinbase Coverage Limits Raise Investor Questions (finance.yahoo.com, 2026-03-27 21:07) - Score: -0.88
vvv Sector Update: Financial Stocks Decline Late Afternoon (finance.yahoo.com, 2026-03-27 21:01) - Score: -0.88
vvv Bitcoin Drops 4.3% as $14 Billion Options Expiry Pressures Market (finance.yahoo.com, 2026-03-27 21:00) - Score: -0.88
vvv Sector Update: Financial Stocks Decline Friday Afternoon (finance.yahoo.com, 2026-03-27 19:04) - Score: -0.87
